In [ ]:
import sys
sys.path.append(r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X')

import csv
import pandas as pd
from pydantic import BaseModel, ValidationError


class DataFrameSchema(BaseModel):
    id_vaga: int


with open(r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X\app\data\processed\vagas_norm.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    data = [row for row in reader]  

data

[{'id_vaga': '30077919',
  'data_anuncio': '2024-08-16',
  'titulo_vaga': 'Analista de Dados - Júnior',
  'titulo_resumo': 'analista-de-dados-junior',
  'faixa_salarial': 'não disponível',
  'empresa_contratante': 'CARTÃO DE TODOS',
  'estado': 'MG',
  'cidade': 'Ipatinga',
  'url': '/vagas/analista-de-dados-junior/30077919/',
  'descricao': 'Para agregar nosso time, suas principais atividades a serem exercidas serão:  Executar atividades de extração, transformação e carregamento (ETL) de dados, assegurando a integridade e precisão das informações processadas. Desenvolver e manter dashboards interativos para visualização e análise de dados, adaptando-os às necessidades dos stakeholders. Criar e monitorar indicadores-chave de performance (KPIs). Automatizar processos para melhorar a eficiência e reduzir erros. Realizar análises estatísticas para gerar insights que apoiem a tomada de decisões e contribuir com valores incrementais. Garantir a entrega dos projetos dentro dos prazos estipul

In [ ]:
from typing import Optional

class SchemaDFVagas(BaseModel):
    id_vaga: str
    data_anuncio: str
    titulo_vaga: str
    titulo_resumo: str
    faixa_salarial: Optional[str]
    empresa_contratante: str
    estado:Optional[str]
    cidade: Optional[str] 
    url: str
    descricao: str
    beneficios: Optional[str]
    regimeContrato: Optional[str]
    regiao: Optional[str]
    perfil_vaga: str
    nivel_cargo: str
    salario: Optional[int]


with open(r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X\app\data\processed\vagas_norm.csv', mode='r', encoding='utf-8') as file:
    dados = csv.DictReader(file)

    # Validando os dados com Pydantic
    dados_validados = []
    linha = 0

    for item in dados:
        linha += 1
        try:
            dados_validados.append(SchemaDFVagas(**item))
        except ValidationError as erro:
            log = erro.errors()[0]
            print(erro)
            print('Erro na Linha:', linha, '| Coluna: ', log['loc'][0], '| Valor: ', log['input'], '| Tipo Esperado:', log['type'])

    list_dict_resul = [item.model_dump() for item in dados_validados]

1 validation error for SchemaDFVagas
salario
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/int_parsing
Erro na Linha: 1 | Coluna:  salario | Valor:   | Tipo Esperado: int_parsing
1 validation error for SchemaDFVagas
salario
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/int_parsing
Erro na Linha: 2 | Coluna:  salario | Valor:   | Tipo Esperado: int_parsing
1 validation error for SchemaDFVagas
salario
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/int_parsing
Erro na Linha: 3 | Coluna:  salario | Valor:   | Tipo Esperado: int_parsing
1 validation error for Sc

In [ ]:
import pandas as pd

df_vagas = pd.read_csv(r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X\app\data\processed\vagas_norm.csv')

lista_ids = ['30077919', '30045699','30600767']

df_ids = pd.DataFrame(lista_ids, columns=['id_vaga'])

df = pd.merge(df_ids, df_vagas, on='id_vaga', how='left')

for _, row in df.iterrows():
    texto = f'''
        {row['titulo_vaga']}
        Perfil: {row['perfil_vaga']}-{row['nivel_cargo']}
        Empresa: {row['empresa_contratante']}
        Local: {row['cidade']} - {row['estado']}
        Anunciada em: {row['data_anuncio']}
        Disponível no link: {row['url']}
        Descrição: {row['descricao']}
    '''

return texto


        Analista de Dados em Excel
        Perfil: analista-pleno
        Empresa: JETER CONSULTING
        Local: São Paulo - SP
        Anunciada em: 2024-10-07
        Disponível no link: /vagas/analista-de-dados-em-excel/30600767/
        Descrição: Com experiência comprovada como Analista de Sistemas.Descrição: Precisará trabalhar de forma sistemática e eficiente, sabendo entender os objetivos e necessidades da colaboração, coletar dados de sistemas internos, excel, banco de dados ou de dados não estruturados, garantir integridade e precisão dos dados coletador e apresentar os desempenhos e resultados.Habilidades técnicas:Liderança;Comunicação assertiva;Excel;Habilidades organizacionais;Comunicação Eficaz.Horário de Trabalho: Segunda à sexta dás 08h às 17h48min.Local: Rua 25 de Março,595
    


In [ ]:
import sys
sys.path.append(r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X')

import os
from typing import List
from fastapi.encoders import jsonable_encoder
from app.services.llm_search import search_vagas
from app.model.api_models import *
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

    
def search_vagas(model_name, cache_file, db_path, input_sentence): 

    # Inicializar o modelo
    try: model = SentenceTransformer(model_name)
    except: {"error": "⛔ Erro ao instanciar o modelo"}

    # Verificar se o cache já existe
    if os.path.exists(cache_file):
        try: sentence_embeddings = np.load(cache_file)
        except: return {"error": "⛔ Erro ao carregar o cache existente do modelo de LLM."}

    else:
        try:
            df = pd.read_csv(db_path)
            #transformar em um dicionário com id_vaga e descricao
            df = df[['id_vaga', 'descricao']].set_index('id_vaga').to_dict()['descricao']

            sentence_embeddings = model.encode(list(df.values()))
            np.save(cache_file, sentence_embeddings)
        except: return {"error": "⛔ Erro ao criar o embeddings da base de dados"}


    # Codificar a frase de entrada (input)
    
    try: input_embedding = model.encode(input_sentence)
    except: return {"error": "⛔ Erro ao realizar o embedding do input_sentence"}

    # Calcular a similaridade entre a frase de entrada e o conjunto de dados
    try:
        similarities = util.cos_sim(input_embedding, sentence_embeddings).cpu().numpy().flatten()  # Garantir que seja um vetor 1D
    except: return {"error": "⛔ Erro ao calcular as similaridades"}

    top_3_indices = np.argsort(similarities)[::-1][:3]
    response = top_3_indices.tolist()

    response =  top_3_indices.to_dict(orient='records')

    print(response)

    return {"success": response}

BASE_DIR = r'C:\Users\RodrigoPintoMesquita\Documents\GitHub\PB_TP_X'

dic_paths = {
    'csv_links': os.path.join(BASE_DIR, r'app\data\raw\links_vagas_catho.csv'),
    'csv_links_indeed' : os.path.join(BASE_DIR, r'app\data\raw\links_vagas_indeed.csv'),
    'folder_htmls': os.path.join(BASE_DIR, r'app\data\html_pages'),
    'csv_vagas_catho': os.path.join(BASE_DIR, r'app\data\raw\vagas_catho.csv'),
    'csv_vagas_indeed' : os.path.join(BASE_DIR, r'app\data\raw\vagas_indeed.csv'),
    'csv_vagas': os.path.join(BASE_DIR, r'app\data\raw\vagas.csv'),
    'csv_vagas_norm': os.path.join(BASE_DIR, r'app\data\processed\vagas_norm.csv'),
    'csv_lista_ferramentas': os.path.join(BASE_DIR, r'app\data\processed\ferramentas.csv'),
    'csv_requisitos': os.path.join(BASE_DIR, r'app\data\processed\requisitos.csv')
}

result = search_vagas(
    model_name = "sentence-transformers/all-mpnet-base-v2", 
    cache_file = "embeddings_cache3.npy", 
    db_path = dic_paths['csv_vagas_norm'],
    input_sentence = 'teste'
)

print(result)